# 6.2. Clean groups data
This notebook creates clean datasets of labgroupids mapped to other groups.

Input:
- Groups equipment sharing cleaning worksheet
- Groups space sharing cleaning worksheet
- Individual processed data

Output:
- Cleaned dataset of labgroupids and groups (labgroupids)
- Cleaned dataset of labgroupids and groups (labgroupids)

In [1]:
# Set-up
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os

In [2]:
# Load data
labs = pd.read_csv(
    config.PROCESSED_DATA / "individual_processed_1.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

share_equip_groups = pd.read_excel(
    config.CLEANING_WORKBOOKS / "groups_cleaning_workbook_final.xlsx",
    sheet_name="share_equip_groups",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

share_space_groups = pd.read_excel(
    config.CLEANING_WORKBOOKS / "groups_cleaning_workbook_final.xlsx",
    sheet_name="share_space_groups",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## (1) Prepare datasets for merging

In [3]:
# Keep only relevant columns of labs dataframe
cols_to_keep = ["labgroupid"] + [col for col in labs.columns if col.startswith("share_space_groups_") or col.startswith("share_equip_groups_")]
labs = labs[cols_to_keep]

In [4]:
# Dataset for merging share_space_groups 

# Number of share_space_groups entries (share_space_groups_1 .. share_space_groups_N)
n_entries_space = max(int(col.split("_")[3]) for col in labs.columns if col.startswith("share_space_groups_") and not col.endswith("_co"))

# Make labs dataframe long format with columns "labgroupid", "entry_number" (1, 2, etc.), "raw_value" (share_space_groups_i), and "comment" (share_space_groups_i_co)
labs_long_share_space = pd.concat([
    labs[["labgroupid", f"share_space_groups_{i}", f"share_space_groups_{i}_co"]]
    .rename(columns={f"share_space_groups_{i}": "raw_value", f"share_space_groups_{i}_co": "comment"})
    .assign(entry_number=i)
    for i in range(1, n_entries_space + 1)
], ignore_index=True)

# Drop empty entries (both raw_value and comment missing)
labs_long_share_space = labs_long_share_space.dropna(subset=["raw_value", "comment"], how="all").reset_index(drop=True)

In [5]:
# Number of share_equip_groups entries (share_equip_groups_1 .. share_equip_groups_N)
n_entries_equip = max(int(col.split("_")[3]) for col in labs.columns if col.startswith("share_equip_groups_") and not col.endswith("_co"))

# Make labs dataframe long format with columns "labgroupid", "entry_number" (1, 2, etc.), "raw_value" (share_equip_groups_i), and "comment" (share_equip_groups_i_co)
labs_long_share_equip = pd.concat([
    labs[["labgroupid", f"share_equip_groups_{i}", f"share_equip_groups_{i}_co"]]
    .rename(columns={f"share_equip_groups_{i}": "raw_value", f"share_equip_groups_{i}_co": "comment"})
    .assign(entry_number=i)
    for i in range(1, n_entries_equip + 1)
], ignore_index=True)

# Drop empty entries
labs_long_share_equip = labs_long_share_equip.dropna(subset=["raw_value", "comment"], how="all").reset_index(drop=True)

In [6]:
# Keep only relevant columns of share_space_groups dataframe
cols_to_keep_space = ["raw_value", "comment", "cleaned_value", "status"]
share_space_groups = share_space_groups[cols_to_keep_space]

In [7]:
# Keep only relevant columns of share_equip_groups dataframe
cols_to_keep_equip = ["raw_value", "comment", "cleaned_value", "status"]
share_equip_groups = share_equip_groups[cols_to_keep_equip]

## (2) Merge datasets to get cleaned rooms

### 2.1 Share space groups

In [8]:
# For each room entry, merge on raw_value + comment to get the cleaned room value
labs_long_share_space = labs_long_share_space.merge(share_space_groups, on=["raw_value", "comment"], how="left", validate="m:1", indicator=True)

# Flag entries not yet in the cleaning workbook, so they can be added and cleaned there
unmatched = labs_long_share_space[labs_long_share_space["_merge"] == "left_only"]
if not unmatched.empty:
    print(f"{len(unmatched)} room entries not found in the cleaning workbook:")
    print(unmatched[["labgroupid", "entry_number", "raw_value", "comment"]].to_string(index=False))

labs_long_share_space = labs_long_share_space.drop(columns="_merge")

In [9]:
# For all entries with several groups reported (separated by comma), split into separate rows
labs_long_share_space = labs_long_share_space.assign(
    group=labs_long_share_space["cleaned_value"].str.split(" / ")).explode("group").reset_index(drop=True)

In [10]:
# Drop all rows where "group" is missing
labs_long_share_space = labs_long_share_space.dropna(subset=["group"]).reset_index(drop=True)

In [11]:
# Create variable "sample_group" which is "group" (as an int) if "group" is a labgroupid in our sample
labgroupids = set(labs["labgroupid"])
group_numeric = pd.to_numeric(labs_long_share_space["group"], errors="coerce")
labs_long_share_space["sample_group"] = group_numeric.where(group_numeric.isin(labgroupids)).astype("Int64")

### 2.2 Share equip groups

In [12]:
# For each group entry, merge on raw_value + comment to get the cleaned room value
labs_long_share_equip = labs_long_share_equip.merge(share_equip_groups, on=["raw_value", "comment"], how="left", validate="m:1", indicator=True)

# Flag entries not yet in the cleaning workbook, so they can be added and cleaned there
unmatched = labs_long_share_equip[labs_long_share_equip["_merge"] == "left_only"]
if not unmatched.empty:
    print(f"{len(unmatched)} room entries not found in the cleaning workbook:")
    print(unmatched[["labgroupid", "entry_number", "raw_value", "comment"]].to_string(index=False))

labs_long_share_equip = labs_long_share_equip.drop(columns="_merge")

In [13]:
# For all entries with several groups reported (separated by comma), split into separate rows
labs_long_share_equip = labs_long_share_equip.assign(
    group=labs_long_share_equip["cleaned_value"].str.split(" / ")).explode("group").reset_index(drop=True)

In [14]:
# Drop all rows where "group" is missing
labs_long_share_equip = labs_long_share_equip.dropna(subset=["group"]).reset_index(drop=True)

In [15]:
# Create variable "sample_group" which is "group" (as an int) if "group" is a labgroupid in our sample
group_numeric = pd.to_numeric(labs_long_share_equip["group"], errors="coerce")
labs_long_share_equip["sample_group"] = group_numeric.where(group_numeric.isin(labgroupids)).astype("Int64")

## (3) Save cleaned dataset

In [16]:
# Keep relevant columns and save cleaned dataset (one row per labgroupid x group entry)

# Share space groups
share_space_groups_clean = labs_long_share_space[["labgroupid", "group", "sample_group"]]
share_space_groups_clean.to_csv(config.CLEAN_DATA / "share_space_groups_cleaned.csv", index=False)

# Share equip groups
share_equip_groups_clean = labs_long_share_equip[["labgroupid", "group", "sample_group"]]
share_equip_groups_clean.to_csv(config.CLEAN_DATA / "share_equip_groups_cleaned.csv", index=False)
